In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/[Research] EEG/[Completed] Tesis/Baseline models/v2/Brain-computer-interfaces-master/notebooks

/content/drive/MyDrive/[Research] EEG/[Completed] Tesis/Baseline models/v2/Brain-computer-interfaces-master/notebooks


In [3]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

In [4]:
%%capture
import math

import numpy as np
import scipy.io as sio
from sklearn.metrics import confusion_matrix

from scripts import ssvep_utils as su

In [5]:
!pip install meegkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.8/90.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.7/121.7 kB 7.8 MB/s eta 0:00:00


In [8]:
data_path = os.path.abspath('../data')
all_segment_data = dict()
all_acc = list()
window_len = 0.1
n_classes = 10
shift_len = 0.1
sample_rate = 256
duration = int(window_len*sample_rate)
flicker_freq = np.array([9.25, 11.25, 13.25, 9.75, 11.75, 13.75,
                       10.25, 12.25, 14.25, 10.75, 12.75, 14.75])

# Load dataset, filter and segment epochs

In [17]:
def get_subject_indepedent(target_subject:int):

  dataset = sio.loadmat(f'{data_path}/s{target_subject+1}.mat')
  eeg = np.array(dataset['eeg'], dtype='float32')

  num_classes = eeg.shape[0]
  n_ch = eeg.shape[1]
  total_trial_len = eeg.shape[2]
  num_trials = eeg.shape[3]

  filtered_data = su.get_filtered_eeg(eeg, 6, 80, 4, sample_rate)
  segmented_data = su.get_segmented_epochs(filtered_data, window_len, shift_len, sample_rate)
  segmented_data = np.expand_dims(segmented_data[:,:,:,0,:],axis = 3)
  segmented_data = np.swapaxes(segmented_data,1,3)

  #Finally reshaping the data into dim [classes*trials*segments X channels X features X 1]
  train_data = segmented_data.reshape(segmented_data.shape[0]*segmented_data.shape[1]*segmented_data.shape[2],segmented_data.shape[3],segmented_data.shape[4])
  dummy_train = np.random.normal(0,1,(segmented_data.shape[0]*segmented_data.shape[1]*segmented_data.shape[2],segmented_data.shape[3],20))
  train_data = np.concatenate((train_data, dummy_train), axis=-1)

  # Deberian ser [0*trials*sessions,1*trials*sessions]
  labels = np.repeat(np.arange(num_classes), segmented_data.shape[1]*segmented_data.shape[2])

  # Defining data
  test_X = train_data.T
  test_Y = labels

  count = 0
  for subject in np.arange(0, 10):
      if subject == target_subject:
        continue
      dataset = sio.loadmat(f'{data_path}/s{subject+1}.mat')
      eeg = np.array(dataset['eeg'], dtype='float32')

      num_classes = eeg.shape[0]
      n_ch = eeg.shape[1]
      total_trial_len = eeg.shape[2]
      num_trials = eeg.shape[3]

      filtered_data = su.get_filtered_eeg(eeg, 6, 80, 4, sample_rate)
      segmented_data = su.get_segmented_epochs(filtered_data, window_len,shift_len, sample_rate)
      segmented_data = np.expand_dims(segmented_data[:,:,:,0,:],axis = 3)
      segmented_data = np.swapaxes(segmented_data,1,3)

      #Finally reshaping the data into dim [classes*trials*segments X channels X features X 1]
      train_data = segmented_data.reshape(segmented_data.shape[0]*segmented_data.shape[1]*segmented_data.shape[2],segmented_data.shape[3],segmented_data.shape[4])
      dummy_train = np.random.normal(0,1,(segmented_data.shape[0]*segmented_data.shape[1]*segmented_data.shape[2],segmented_data.shape[3],20))
      train_data = np.concatenate((train_data, dummy_train), axis=-1)

      # Deberian ser [0*trials*sessions,1*trials*sessions]
      labels = np.repeat(np.arange(num_classes), segmented_data.shape[1]*segmented_data.shape[2])

      if count == 0:
        train_X = train_data.T
        train_Y = labels
      else:
        # Merging arrays
        train_X = np.concatenate((train_X,train_data.T),axis = -1)
        train_Y = np.concatenate((train_Y,labels),axis = -1)

      count = count + 1

  return train_X,train_Y,test_X,test_Y


# Perform CCA on the segmented epochs

In [18]:
from meegkit.trca import TRCA

for i in range(10):
  is_ensemble = True
  sfreq = 256
  filterbank = [[(6, 90), (4, 100)],  # passband, stopband freqs [(Wp), (Ws)]
                [(14, 90), (10, 100)],
                [(22, 90), (16, 100)],
                [(30, 90), (24, 100)],
                [(38, 90), (32, 100)],
                [(46, 90), (40, 100)],
                [(54, 90), (48, 100)]]

  trca = TRCA(sfreq, filterbank, is_ensemble)

  train_X,train_Y,test_X,test_Y = get_subject_indepedent(i)

  # Train
  trca.fit(train_X, train_Y)

  # Test
  estimated = trca.predict(test_X)

  # Evaluation of the performance for this fold (accuracy and ITR)
  is_correct = estimated == test_Y
  accs = np.mean(is_correct) * 100
  print(accs)

20.555555555555554
7.222222222222221
27.22222222222222
19.444444444444446
13.88888888888889
33.33333333333333
28.333333333333332
34.44444444444444
13.333333333333334
23.333333333333332
